## Imports

In [ ]:
import bokeh.plotting as bpl
import cv2
import holoviews as hv
from IPython import get_ipython
import matplotlib.pyplot as plt
import numpy as np
import os

try:
    cv2.setNumThreads(0)
except():
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

import caiman as cm
from caiman.source_extraction.cnmf import cnmf

bpl.output_notebook()
hv.notebook_extension('bokeh')

## Load saved results

In [ ]:
results_file = "/media/toor/SeagateClean/Mouse1636/Round1/Linear/2024-05-27T13_27_43/caiman_final/caiman_results.hdf5" # Place path to results here

cnmf_fit = cnmf.load_CNMF(results_file,
                            n_processes=1, 
                            dview=None)
correlation_image = cnmf_fit.estimates.Cn

# Yr, dims, num_frames = cm.load_memmap(cnmf_fit.mmap_file)
# images = np.reshape(Yr.T, [num_frames] + list(dims), order="F")

# motion_corrected = cm.load(
#     os.path.join(
#         os.path.dirname(results_file), 
#         "motion_correction_comparison_final2.avi",
#     )
# )
print(f"Successfully loaded data.")

## Visualizing results
We've sorted our components into accepted/rejected, so it's time to start looking at the results! Caiman provides many built-in `estimates` methods for visualizaing results. 

In [ ]:
# Play motion corrected video
# motion_corrected.play(fr=12, magnification=2, offset=0)

### All contours
We have already used `plot_contours_nb()`, but if we provide it the `idx` keyword it will split the view into accepted and rejected components.

In [ ]:
n_acc = len(cnmf_fit.estimates.idx_components)
n_rej = len(cnmf_fit.estimates.idx_components_bad)
print(f"Accepted: {n_acc}  |  Rejected: {n_rej}  |  Total: {n_acc + n_rej}")

cnmf_fit.estimates.plot_contours_nb(img=correlation_image, 
                                      idx=cnmf_fit.estimates.idx_components);

### View individual spatial/temporal components
One of the most useful visualization tools is `nb_view_components()`, which lets you scroll through individiual spatial and temporal components. This tool also displays the values of the three evaluation criteria for each component, which can be useful if you feel you need to change your evaluation criteria and re-run `evaluate_components()`. Perhaps you have too many false negatives and want to lower your SNR threshold. 

In [ ]:
# # view accepted components
# cnmf_fit.estimates.nb_view_components(img=correlation_image, 
#                                         idx=cnmf_fit.estimates.idx_components,
#                                         cmap='gray',
#                                         denoised_color='red',
#                                     );

The above shows the raw traces (`C+YrA`) by default, but you can superimpose the denoised traces from `C` if you add a color to the `denoised_color` parameter. As always in Jupyter, if you are unsure how a method works, you can enter `cnmf_refit.estimates.nb_view_components?` in a new cell to get the documentation for the method. 

We can also view the rejected compoonents:

In [ ]:
# # rejected components
# if len(cnmf_fit.estimates.idx_components_bad) > 0:
#     cnmf_fit.estimates.nb_view_components(
#         img=correlation_image, 
#         idx=cnmf_fit.estimates.idx_components_bad[::8], 
#         cmap='gray',
#         denoised_color='red',
#     )
# else:
#     print("No components were rejected.")

> One legacy from Matlab with these plotters is that they use one-based indexing when showing `Neuron number`. This can make it confusing when comparing to your own plotting results which will use zero-based indexing.

### Building your own visualizations
> This is a slightly more advanced section that you can safely skip your first time through. 

There are many custom visualizations you can build yourself based on the estimates generated from Caiman. For example, what if you wanted to plot the spatial footprint of a neuron with the contour superimposed?  The contours of the spatial footprints are in `estimates.coordinates`, which is a list of dictionaries corresponding to each component. The dictionary includes a `coordinates` field that contains the x,y coordinates of the contour. Here we'll show how to plot this contour superimposed on the corresponding spatial footprint in `A`. 

The following extracts all of the contours of the accepted components into a list from the estimates (it puts them in a list because they are not all the same size):

In [ ]:
idx_accepted = cnmf_fit.estimates.idx_components
all_contour_coords = [cnmf_fit.estimates.coordinates[idx]['coordinates'] for idx in idx_accepted]

Each footprint in `A` is stored as a compressed sparse column array. We can convert it to a dense array with `toarray()`, and plot the contour and footprint together:

In [ ]:
idx_to_plot = 1
component_number = idx_accepted[idx_to_plot]
component_contour = all_contour_coords[idx_to_plot]
dims = cnmf_fit.dims
component_footprint = np.reshape(
    cnmf_fit.estimates.A[:, component_number].toarray(),
    dims,
    order='F',
)

In [ ]:
plt.figure(); 
plt.imshow(component_footprint, cmap='gray');
plt.plot(component_contour[:, 0], 
         component_contour[:, 1], 
         color='pink', 
         linewidth=2)
plt.title(f'Footprint/Contour {component_number}');

# A few final things
We have extracted the calcium traces `C`, spatial footprints `A`, and estimated spike counts `S`, which is the main goal with CNMF. But there are a few important things remaining.

## Extract $\Delta F/F$ values
So far our calcium traces are in arbitrary units. It is more common to report the calcium fluorescence relative to some baseline value $F_0$:

$$
\Delta F/F = (F(t)-F_0)/F_0
$$

Traditionally, the baseline value was calculated during some initial period of queiscience (e.g., in the visual system, when the animal was sitting in the dark). However, it is problematic to make any assumptions about a "quiet" baseline period, so researchers have started to use a moving average to calculate $F_0$. This is also what Caiman does.

More specifically, in Caiman the baseline is a running percentile calculated over a `frames_window` moving window. You can calculate $\Delta F/F$ using raw traces or the denoised traces in C (this is toggled using the `use_residuals` argument).

We can plot a temporal trace of a dff with time. In the following, we'll generate a `frame_time` variable for plotting. This is an  idealized time representation using `linspace()`, which assumes that images were acquired at a constant frequency throughout the session. In practice your hardware probably returns the time at which each frame was acquired, so you can use that information in your own analysis if you have it. 

In [ ]:
# plot F_dff
idx_to_plot = 1
idx_accepted = cnmf_fit.estimates.idx_components
component_number = idx_accepted[idx_to_plot]
f, ax = plt.subplots(figsize=(7,2))
ax.plot(np.arange(cnmf_fit.estimates.F_dff.shape[1]) / 10, 
        cnmf_fit.estimates.F_dff[component_number, :], 
        linewidth=0.5,
        color='k');
ax.set_xlabel('Time (s)')
ax.set_ylabel('$\Delta F/F$')
ax.set_title(f"$\Delta F/F$ for unit {component_number}");
plt.tight_layout()

<div class="alert alert-info" markdown="1">
    <h2 style="margin-top: 0;">More on $\Delta F/F$</h2>  
The above discussion of dff leaves out some details about it is actually calculated in Caiman. In practice, Caiman locally projects the model of the background activity to the spatial footprint of the neuron, and adds a moving percentile of this projected trace to the denominator as an additional normalizing term (see the discussion on the background model below). This acts as an empirical fudge factor that can keep $\Delta F/F$ from getting too large for very small values of $F_0$ (especially when using denoised traces). Users may prefer to use the more traditional equation directly. Thanks to Peter Rupprecht for a helpful discussion of this topic. 
</div>

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.gridspec as gridspec

# ── parameters ──────────────────────────────────────────────────────────────
component_number   = cnmf_fit.estimates.idx_components[-1]  # first accepted component
zoom_pad           = 30     # pixels of padding around the contour bounding box
DS_RATIO           = 0.2    # temporal downsampling used when saving the AVI
embed_limit_mb     = 200    # raise if animation is still getting truncated
# ────────────────────────────────────────────────────────────────────────────

plt.rcParams['animation.embed_limit'] = embed_limit_mb
frame_stride = round(1 / DS_RATIO)


component_contour = cnmf_fit.estimates.coordinates[component_number]['coordinates']

# CaImAn inserts NaNs as segment separators — drop them before computing bounds
cx, cy   = component_contour[:, 0], component_contour[:, 1]
valid    = np.isfinite(cx) & np.isfinite(cy)
cx_valid, cy_valid = cx[valid], cy[valid]
x0, x1  = cx_valid.min() - zoom_pad, cx_valid.max() + zoom_pad
y0, y1  = cy_valid.min() - zoom_pad, cy_valid.max() + zoom_pad

# evaluation metrics for this component
r_val = cnmf_fit.estimates.r_values[component_number]
snr   = cnmf_fit.estimates.SNR_comp[component_number]
cnn   = cnmf_fit.estimates.cnn_preds[component_number] if cnmf_fit.estimates.cnn_preds is not None else float('nan')
metrics_str = f'r={r_val:.2f}  SNR={snr:.2f}  CNN={cnn:.2f}'

# load all frames from the video
video_path = os.path.join(os.path.dirname(results_file), "motion_correction_comparison_final2.avi")
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0

frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    frames.append(gray)
cap.release()
frames = np.array(frames)
if frames.ndim == 0 or frames.shape[0] == 0:
    raise FileNotFoundError(f"No frames loaded from {video_path!r} — check that the file exists.")
if frames.ndim == 3:
    frames = frames[:, :, frames.shape[2] // 2:]  # keep right half (motion-corrected)

# video frame i → recording frame i * frame_stride
trace         = cnmf_fit.estimates.F_dff[component_number, :]
original_fps  = fps / DS_RATIO
n_anim_frames = min(len(frames), len(trace) // frame_stride)

trace_clip = trace[:n_anim_frames * frame_stride]
time_axis  = np.arange(len(trace_clip)) / original_fps

print(f"Video: {len(frames)} frames at {fps:.1f} fps  |  "
      f"Recording: {len(trace)} frames at {original_fps:.1f} fps  |  "
      f"Animating {n_anim_frames} video frames")

# ── build figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 7))
gs  = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.45, wspace=0.05)

ax_zoom  = fig.add_subplot(gs[0, 0])
ax_fov   = fig.add_subplot(gs[0, 1])
ax_trace = fig.add_subplot(gs[1, :])

vmin, vmax = np.percentile(frames, [1, 99])

# zoomed panel
im_zoom = ax_zoom.imshow(frames[0], cmap='gray', vmin=vmin, vmax=vmax)
ax_zoom.plot(cx, cy, color='cyan', linewidth=1.5)
ax_zoom.set_xlim(x0, x1)
ax_zoom.set_ylim(y1, y0)
ax_zoom.set_title(f'Component {component_number} (zoom)  t = 0.00 s\n{metrics_str}', fontsize=9)
ax_zoom.axis('off')

# full FOV panel
im_fov = ax_fov.imshow(frames[0], cmap='gray', vmin=vmin, vmax=vmax)
ax_fov.plot(cx, cy, color='cyan', linewidth=1.5)
ax_fov.set_title('Full FOV', fontsize=9)
ax_fov.axis('off')

# trace panel
ax_trace.plot(time_axis, trace_clip, color='k', linewidth=0.6)
vline = ax_trace.axvline(x=time_axis[0], color='r', linewidth=1)
ax_trace.set_xlabel('Time (s)')
ax_trace.set_ylabel('ΔF/F')
ax_trace.set_title(f'Unit {component_number}', fontsize=9)

def _update(i):
    trace_idx = i * frame_stride
    t = time_axis[trace_idx]
    im_zoom.set_data(frames[i])
    im_fov.set_data(frames[i])
    ax_zoom.set_title(f'Component {component_number} (zoom)  t = {t:.2f} s\n{metrics_str}', fontsize=9)
    vline.set_xdata([t])
    return im_zoom, im_fov, vline

anim = FuncAnimation(fig, _update, frames=n_anim_frames,
                     interval=1000 / fps, blit=True)
plt.close()
HTML(anim.to_jshtml())

## Rejected Component video + trace
Plays a short clip from the motion-corrected video with the accepted component's contour overlaid, and shows the ΔF/F trace below with a red cursor tracking the current frame.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.gridspec as gridspec

# ── parameters ──────────────────────────────────────────────────────────────
component_number   = cnmf_fit.estimates.idx_components_bad[46]  # first rejected component
zoom_pad           = 30     # pixels of padding around the contour bounding box
DS_RATIO           = 0.2    # temporal downsampling used when saving the AVI
embed_limit_mb     = 200    # raise if animation is still getting truncated
# ────────────────────────────────────────────────────────────────────────────

plt.rcParams['animation.embed_limit'] = embed_limit_mb
frame_stride = round(1 / DS_RATIO)


component_contour = cnmf_fit.estimates.coordinates[component_number]['coordinates']

# CaImAn inserts NaNs as segment separators — drop them before computing bounds
cx, cy   = component_contour[:, 0], component_contour[:, 1]
valid    = np.isfinite(cx) & np.isfinite(cy)
cx_valid, cy_valid = cx[valid], cy[valid]
x0, x1  = cx_valid.min() - zoom_pad, cx_valid.max() + zoom_pad
y0, y1  = cy_valid.min() - zoom_pad, cy_valid.max() + zoom_pad

# evaluation metrics for this component
r_val = cnmf_fit.estimates.r_values[component_number]
snr   = cnmf_fit.estimates.SNR_comp[component_number]
cnn   = cnmf_fit.estimates.cnn_preds[component_number] if cnmf_fit.estimates.cnn_preds is not None else float('nan')
metrics_str = f'r={r_val:.2f}  SNR={snr:.2f}  CNN={cnn:.2f}'

# load all frames from the video
video_path = os.path.join(os.path.dirname(results_file), "motion_correction_comparison_final2.avi")
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0

frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    frames.append(gray)
cap.release()
frames = np.array(frames)
if frames.ndim == 0 or frames.shape[0] == 0:
    raise FileNotFoundError(f"No frames loaded from {video_path!r} — check that the file exists.")
if frames.ndim == 3:
    frames = frames[:, :, frames.shape[2] // 2:]  # keep right half (motion-corrected)

# video frame i → recording frame i * frame_stride
trace         = cnmf_fit.estimates.F_dff[component_number, :]
original_fps  = fps / DS_RATIO
n_anim_frames = min(len(frames), len(trace) // frame_stride)

trace_clip = trace[:n_anim_frames * frame_stride]
time_axis  = np.arange(len(trace_clip)) / original_fps

print(f"Video: {len(frames)} frames at {fps:.1f} fps  |  "
      f"Recording: {len(trace)} frames at {original_fps:.1f} fps  |  "
      f"Animating {n_anim_frames} video frames")

# ── build figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 7))
gs  = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.45, wspace=0.05)

ax_zoom  = fig.add_subplot(gs[0, 0])
ax_fov   = fig.add_subplot(gs[0, 1])
ax_trace = fig.add_subplot(gs[1, :])

vmin, vmax = np.percentile(frames, [1, 99])

# zoomed panel
im_zoom = ax_zoom.imshow(frames[0], cmap='gray', vmin=vmin, vmax=vmax)
ax_zoom.plot(cx, cy, color='cyan', linewidth=1.5)
ax_zoom.set_xlim(x0, x1)
ax_zoom.set_ylim(y1, y0)
ax_zoom.set_title(f'Component {component_number} (zoom)  t = 0.00 s\n{metrics_str}', fontsize=9)
ax_zoom.axis('off')

# full FOV panel
im_fov = ax_fov.imshow(frames[0], cmap='gray', vmin=vmin, vmax=vmax)
ax_fov.plot(cx, cy, color='cyan', linewidth=1.5)
ax_fov.set_title('Full FOV', fontsize=9)
ax_fov.axis('off')

# trace panel
ax_trace.plot(time_axis, trace_clip, color='k', linewidth=0.6)
vline = ax_trace.axvline(x=time_axis[0], color='r', linewidth=1)
ax_trace.set_xlabel('Time (s)')
ax_trace.set_ylabel('ΔF/F')
ax_trace.set_title(f'Unit {component_number}', fontsize=9)

def _update(i):
    trace_idx = i * frame_stride
    t = time_axis[trace_idx]
    im_zoom.set_data(frames[i])
    im_fov.set_data(frames[i])
    ax_zoom.set_title(f'Component {component_number} (zoom)  t = {t:.2f} s\n{metrics_str}', fontsize=9)
    vline.set_xdata([t])
    return im_zoom, im_fov, vline

anim = FuncAnimation(fig, _update, frames=n_anim_frames,
                     interval=1000 / fps, blit=True)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
import pandas as pd

idx_accepted = cnmf_fit.estimates.idx_components
idx_rejected = cnmf_fit.estimates.idx_components_bad

# ── figure: contour overlay with IDs ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(correlation_image, cmap='gray')

for color, indices in [('lime', idx_accepted), ('red', idx_rejected)]:
    for idx in indices:
        coords = cnmf_fit.estimates.coordinates[idx]['coordinates']
        ax.plot(coords[:, 0], coords[:, 1], color=color, linewidth=0.8, alpha=0.8)
        valid = np.isfinite(coords[:, 0]) & np.isfinite(coords[:, 1])
        cx = coords[valid, 0].mean()
        cy = coords[valid, 1].mean()
        ax.text(cx, cy, str(idx), color=color, fontsize=10,
                ha='center', va='center', fontweight='bold')

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color='lime', linewidth=1.5, label=f'Accepted ({len(idx_accepted)})'),
    Line2D([0], [0], color='red',  linewidth=1.5, label=f'Rejected ({len(idx_rejected)})'),
], loc='upper right', framealpha=0.7)

ax.set_title('Accepted vs Rejected Components')
ax.axis('off')
plt.tight_layout()
plt.show()

# ── evaluation criteria table ────────────────────────────────────────────────
cnn_preds = cnmf_fit.estimates.cnn_preds

rows = []
for status, indices in [('accepted', idx_accepted), ('rejected', idx_rejected)]:
    for idx in indices:
        rows.append({
            'id':     idx,
            'status': status,
            'r_val':  cnmf_fit.estimates.r_values[idx],
            'SNR':    cnmf_fit.estimates.SNR_comp[idx],
            'CNN':    cnn_preds[idx] if cnn_preds is not None else float('nan'),
        })

df = (pd.DataFrame(rows)
        .sort_values('id')
        .set_index('id')
        .round(3))

df.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]},
]).apply(lambda col: [
    'background-color: #1a3a1a; color: lime' if v == 'accepted'
    else 'background-color: #3a1a1a; color: red'
    for v in col
], subset=['status'])

## Threshold Exploration

Adjust `NEW_THRESHOLDS` and re-run the cell below to instantly see how the accepted/rejected split changes — no need to re-run CaImAn.

**Acceptance logic** (same as CaImAn's `evaluate_components`):
- passes **at least one** upper threshold: `SNR ≥ min_SNR` OR `r_val ≥ rval_thr` OR `CNN ≥ cnn_thr`
- AND passes **all** lower bounds: `SNR ≥ SNR_lowest` AND `r_val ≥ rval_lowest` AND `CNN ≥ cnn_lowest`

The 3D scatter shows every component coloured by its status under the **new** thresholds. Threshold planes (blue = SNR, orange = r_value, purple = CNN) mark the upper cutoffs.

In [ ]:
import plotly.graph_objects as go

# ── adjustable thresholds — edit these and re-run ────────────────────────────
NEW_THRESHOLDS = dict(
    min_SNR     = 2.5,    # upper SNR threshold
    rval_thr    = 0.8,   # upper r_value threshold
    cnn_thr     = 0.95,   # upper CNN threshold
    SNR_lowest  = 1.8,    # lower SNR bound
    rval_lowest = 0.65,    # lower r_value bound
    cnn_lowest  = 0.2,    # lower CNN bound
)

# ── apply threshold logic ─────────────────────────────────────────────────────
def apply_thresholds(estimates, thr):
    n   = estimates.A.shape[1]
    rv  = estimates.r_values
    snr = estimates.SNR_comp
    cnn = estimates.cnn_preds if estimates.cnn_preds is not None else np.ones(n)
    upper = (snr >= thr['min_SNR']) | (rv >= thr['rval_thr']) | (cnn >= thr['cnn_thr'])
    lower = (snr >= thr['SNR_lowest']) & (rv >= thr['rval_lowest']) & (cnn >= thr['cnn_lowest'])
    return np.where(upper & lower)[0], np.where(~(upper & lower))[0]

new_acc, new_rej = apply_thresholds(cnmf_fit.estimates, NEW_THRESHOLDS)
orig_acc = np.array(cnmf_fit.estimates.idx_components)
orig_rej = np.array(cnmf_fit.estimates.idx_components_bad)

gained = sorted(set(new_acc.tolist()) - set(orig_acc.tolist()))
lost   = sorted(set(orig_acc.tolist()) - set(new_acc.tolist()))
print(f"Original  →  accepted: {len(orig_acc):3d}  |  rejected: {len(orig_rej)}")
print(f"New       →  accepted: {len(new_acc):3d}  |  rejected: {len(new_rej)}")
print(f"Gained: {gained if gained else 'none'}")
print(f"Lost:   {lost   if lost   else 'none'}")

# ── build plotly figure ───────────────────────────────────────────────────────
n     = cnmf_fit.estimates.A.shape[1]
rv    = cnmf_fit.estimates.r_values
snr   = cnmf_fit.estimates.SNR_comp
cnn_p = cnmf_fit.estimates.cnn_preds
if cnn_p is None:
    cnn_p = np.ones(n)

valid     = np.isfinite(rv) & np.isfinite(snr) & np.isfinite(cnn_p)
all_ids   = np.arange(n)
acc_set   = set(new_acc.tolist())

fig = go.Figure()

for ids_set, color, label in [
    (acc_set,              '#2ca02c', f'accepted ({len(new_acc)})'),
    (set(all_ids) - acc_set, '#d62728', f'rejected ({len(new_rej)})'),
]:
    mask = valid & np.array([i in ids_set for i in all_ids])
    comp_ids = all_ids[mask]
    hover = [
        f"ID: {i}<br>r_val: {rv[i]:.3f}<br>SNR: {snr[i]:.2f}<br>CNN: {cnn_p[i]:.3f}"
        for i in comp_ids
    ]
    fig.add_trace(go.Scatter3d(
        x=rv[mask], y=snr[mask], z=cnn_p[mask],
        mode='markers',
        marker=dict(size=5, color=color, opacity=0.75),
        name=label,
        text=hover,
        hovertemplate='%{text}<extra></extra>',
    ))

# threshold planes (semi-transparent surfaces)
r_lo, r_hi = rv[valid].min(),    rv[valid].max()
s_lo, s_hi = snr[valid].min(),   snr[valid].max()
c_lo, c_hi = cnn_p[valid].min(), cnn_p[valid].max()

def _surface(x, y, z, color, name):
    return go.Surface(
        x=x, y=y, z=z,
        colorscale=[[0, color], [1, color]],
        opacity=0.15,
        showscale=False,
        name=name,
        hoverinfo='skip',
    )

# SNR plane (blue) — constant y
if s_lo <= NEW_THRESHOLDS['min_SNR'] <= s_hi:
    X, Z = np.meshgrid([r_lo, r_hi], [c_lo, c_hi])
    Y    = np.full_like(X, NEW_THRESHOLDS['min_SNR'])
    fig.add_trace(_surface(X, Y, Z, 'steelblue', f"SNR={NEW_THRESHOLDS['min_SNR']}"))

# r_value plane (orange) — constant x
if r_lo <= NEW_THRESHOLDS['rval_thr'] <= r_hi:
    Y, Z = np.meshgrid([s_lo, s_hi], [c_lo, c_hi])
    X    = np.full_like(Y, NEW_THRESHOLDS['rval_thr'])
    fig.add_trace(_surface(X, Y, Z, 'orange', f"r={NEW_THRESHOLDS['rval_thr']}"))

# CNN plane (purple) — constant z
if c_lo <= NEW_THRESHOLDS['cnn_thr'] <= c_hi:
    X, Y = np.meshgrid([r_lo, r_hi], [s_lo, s_hi])
    Z    = np.full_like(X, NEW_THRESHOLDS['cnn_thr'])
    fig.add_trace(_surface(X, Y, Z, 'purple', f"CNN={NEW_THRESHOLDS['cnn_thr']}"))

fig.update_layout(
    scene=dict(
        xaxis_title='r value',
        yaxis_title='SNR',
        zaxis_title='CNN',
    ),
    title=(
        f"Threshold exploration — "
        f"SNR≥{NEW_THRESHOLDS['min_SNR']}  "
        f"r≥{NEW_THRESHOLDS['rval_thr']}  "
        f"CNN≥{NEW_THRESHOLDS['cnn_thr']}<br>"
        f"<sup>Accepted: {len(new_acc)}  "
        f"(+{len(gained)} / -{len(lost)} vs original)</sup>"
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=60, b=0),
    width=800, height=650,
)
fig.show()


In [ ]:
# Re-run the threshold exploration cell first to define new_acc / new_rej.
# This cell reproduces the contour map and score table using the NEW thresholds.

import pandas as pd
from caiman.utils.visualization import get_contours

# ensure coordinates are populated
if cnmf_fit.estimates.coordinates is None:
    cnmf_fit.estimates.coordinates = get_contours(
        cnmf_fit.estimates.A, correlation_image.shape)

gained_set = set(new_acc.tolist()) - set(orig_acc.tolist())
lost_set   = set(orig_acc.tolist()) - set(new_acc.tolist())

# ── contour overlay ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(correlation_image, cmap='gray')

for color, indices, label in [
    ('lime',   new_acc, f'Accepted ({len(new_acc)})'),
    ('red',    new_rej, f'Rejected ({len(new_rej)})'),
]:
    for idx in indices:
        coords = cnmf_fit.estimates.coordinates[idx]['coordinates']
        valid  = np.isfinite(coords[:, 0]) & np.isfinite(coords[:, 1])
        ax.plot(coords[:, 0], coords[:, 1], color=color, linewidth=0.8, alpha=0.8)
        cx = coords[valid, 0].mean()
        cy = coords[valid, 1].mean()

        # highlight components that changed status
        if idx in gained_set:
            ax.plot(coords[:, 0], coords[:, 1], color='cyan',
                    linewidth=2.5, alpha=0.9, zorder=3)
            ax.text(cx, cy, str(idx), color='cyan', fontsize=10,
                    ha='center', va='center', fontweight='bold')
        elif idx in lost_set:
            ax.plot(coords[:, 0], coords[:, 1], color='yellow',
                    linewidth=2.5, alpha=0.9, zorder=3)
            ax.text(cx, cy, str(idx), color='yellow', fontsize=10,
                    ha='center', va='center', fontweight='bold')
        else:
            ax.text(cx, cy, str(idx), color=color, fontsize=10,
                    ha='center', va='center', fontweight='bold')

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0],[0], color='lime',   linewidth=1.5, label=f'Accepted ({len(new_acc)})'),
    Line2D([0],[0], color='red',    linewidth=1.5, label=f'Rejected ({len(new_rej)})'),
]
if gained_set:
    legend_handles.append(
        Line2D([0],[0], color='cyan',   linewidth=2.5,
               label=f'Newly accepted ({len(gained_set)}): {sorted(gained_set)}'))
if lost_set:
    legend_handles.append(
        Line2D([0],[0], color='yellow', linewidth=2.5,
               label=f'Newly rejected ({len(lost_set)}): {sorted(lost_set)}'))

# ax.legend(handles=legend_handles, loc='upper right', framealpha=0.8, fontsize=8)
ax.set_title(
    f"Contours under new thresholds  "
    f"(SNR≥{NEW_THRESHOLDS['min_SNR']}  "
    f"r≥{NEW_THRESHOLDS['rval_thr']}  "
    f"CNN≥{NEW_THRESHOLDS['cnn_thr']})"
)
ax.axis('off')
plt.tight_layout()
plt.show()

# ── evaluation table ──────────────────────────────────────────────────────────
cnn_preds = cnmf_fit.estimates.cnn_preds
rows = []
for status, indices in [('accepted', new_acc), ('rejected', new_rej)]:
    for idx in indices:
        changed = 'gained' if idx in gained_set else ('lost' if idx in lost_set else '')
        rows.append({
            'id':      idx,
            'status':  status,
            'changed': changed,
            'r_val':   cnmf_fit.estimates.r_values[idx],
            'SNR':     cnmf_fit.estimates.SNR_comp[idx],
            'CNN':     cnn_preds[idx] if cnn_preds is not None else float('nan'),
        })

df = (pd.DataFrame(rows)
        .sort_values('id')
        .set_index('id')
        .round(3))

def _row_style(row):
    if row['changed'] == 'gained':
        base = 'background-color: #0d2b2b; color: cyan'
    elif row['changed'] == 'lost':
        base = 'background-color: #2b2b00; color: yellow'
    elif row['status'] == 'accepted':
        base = 'background-color: #1a3a1a; color: lime'
    else:
        base = 'background-color: #3a1a1a; color: red'
    return [base] * len(row)

df.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]},
]).apply(_row_style, axis=1)


In [ ]:
# Accepted components under NEW thresholds — same interactive contour plot style.
# Requires the threshold exploration cell (cell 30) to have been run first.

cnmf_fit.estimates.plot_contours_nb(img=correlation_image, idx=new_acc.tolist());